In [8]:
from dotenv import load_dotenv
import os
import pandas as pd

In [3]:
load_dotenv()

True

In [5]:
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
# response = client.embeddings.create(model="text-embedding-3-small", input="안녕하세요 임베딩 테스트입니다.")
# len(response.data[0].embedding)

1536

In [9]:
# 문서
data = pd.read_csv("../data/11-1_뉴스정제.csv")

In [11]:
doc_list = data['정제본문'].head(30).tolist()

In [12]:
import time
# 벡터 문서 -> 임베딩

start = time.time() # 시작 시간
doc_response = client.embeddings.create(model="text-embedding-3-small", input=doc_list)
api_time = time.time() - start
print(api_time)

0.7158634662628174


In [13]:
# 1. 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
doc_response.data[0].embedding
# doc_response.data

[-0.020660400390625,
 -0.0028591156005859375,
 0.034210205078125,
 0.034271240234375,
 0.01126861572265625,
 0.0037384033203125,
 0.0038433074951171875,
 0.0570068359375,
 -0.0007305145263671875,
 -0.059722900390625,
 -0.034515380859375,
 -0.0190277099609375,
 0.0014085769653320312,
 -0.04046630859375,
 0.0245513916015625,
 -0.011505126953125,
 -0.004791259765625,
 -0.0009984970092773438,
 0.0478515625,
 -0.012451171875,
 -0.0214691162109375,
 -0.02880859375,
 0.01332855224609375,
 0.006488800048828125,
 -0.007358551025390625,
 -0.018646240234375,
 0.01395416259765625,
 0.0479736328125,
 -0.0010051727294921875,
 -0.01537322998046875,
 -0.03497314453125,
 -0.03692626953125,
 0.0318603515625,
 0.015472412109375,
 0.0149078369140625,
 0.0260772705078125,
 0.0093994140625,
 0.0213623046875,
 0.0309295654296875,
 -0.00678253173828125,
 0.03057861328125,
 -0.006877899169921875,
 0.003173828125,
 0.004497528076171875,
 0.0196990966796875,
 0.0259857177734375,
 -0.0028095245361328125,
 0.00177

In [23]:
# api_doc_vecs = []

# for i in doc_response.data:
#     api_doc_vecs.append(i.embedding)

# 위 내용을 리스트 comprehension으로 한줄로 나타내려면

api_doc_vecs = [d.embedding for d in doc_response.data]

In [21]:
# 질문도 벡터로!

query = "우리 집 아파트는 언제 오르나요?"
api_response = client.embeddings.create(model="text-embedding-3-small", input=query)
api_response

CreateEmbeddingResponse(data=[Embedding(embedding=[-0.0177764892578125, -0.0654296875, 0.022125244140625, -0.022918701171875, -0.003833770751953125, -0.01068115234375, -0.04693603515625, -0.0345458984375, -0.00611114501953125, -0.050323486328125, -0.0230865478515625, 0.004955291748046875, 0.00421905517578125, -0.03619384765625, 0.0015506744384765625, 0.037445068359375, -0.004276275634765625, 0.0189971923828125, -0.050445556640625, -0.01812744140625, 0.0236053466796875, -0.0222625732421875, -0.04254150390625, -0.0389404296875, -0.01425933837890625, -0.0244598388671875, 0.03271484375, 0.00803375244140625, 0.0078582763671875, 0.0255889892578125, -0.041351318359375, -0.040557861328125, 0.056182861328125, 0.056884765625, -0.0070037841796875, 0.018798828125, 0.026580810546875, -0.04632568359375, -0.0255279541015625, -0.041015625, -0.004978179931640625, 0.0022430419921875, 0.0243072509765625, 0.04803466796875, -0.0159912109375, 0.076416015625, -0.0292205810546875, 0.0228271484375, 0.012252807

In [24]:
api_query_vec = [api_response.data[0].embedding] # 리스트안에 넣는 이유: 위의 api_doc_vecs와 형태 맞추려고

In [28]:
api_sim = cosine_similarity(api_query_vec, api_doc_vecs)[0]
api_sim

array([ 0.12344549,  0.07612205,  0.17595955,  0.15439471,  0.07907581,
        0.05586888,  0.30266432,  0.00196064,  0.15796446,  0.04465145,
        0.0444624 ,  0.04524953,  0.06272936,  0.0174554 ,  0.00260002,
        0.05671117,  0.11359961,  0.14136845,  0.05722282,  0.11369239,
        0.14999793,  0.09578419,  0.0868522 ,  0.17398512,  0.16228086,
       -0.01307989,  0.22468804,  0.03279508,  0.00966807,  0.19379186])

In [30]:
api_rank = api_sim.argsort(descending=True)
api_rank

array([ 6, 26, 29,  2, 23, 24,  8,  3, 20, 17,  0, 19, 16, 21, 22,  4,  1,
       12, 18, 15,  5, 11,  9, 10, 27, 13, 28, 14,  7, 25])

In [32]:
for rank, index in enumerate(api_rank):
    print(
        rank + 1,
        "문서 인덱스:", index.item(),
        "유사도:", api_sim[index].item()
    )

1 문서 인덱스: 6 유사도: 0.30266431994767495
2 문서 인덱스: 26 유사도: 0.22468804164147804
3 문서 인덱스: 29 유사도: 0.19379185991294906
4 문서 인덱스: 2 유사도: 0.17595954854677454
5 문서 인덱스: 23 유사도: 0.1739851159959176
6 문서 인덱스: 24 유사도: 0.16228086302750833
7 문서 인덱스: 8 유사도: 0.15796446412112852
8 문서 인덱스: 3 유사도: 0.15439471443374125
9 문서 인덱스: 20 유사도: 0.14999792500817793
10 문서 인덱스: 17 유사도: 0.14136845447289992
11 문서 인덱스: 0 유사도: 0.123445491971889
12 문서 인덱스: 19 유사도: 0.11369239135509021
13 문서 인덱스: 16 유사도: 0.11359961107826039
14 문서 인덱스: 21 유사도: 0.09578419296868522
15 문서 인덱스: 22 유사도: 0.08685219797241757
16 문서 인덱스: 4 유사도: 0.07907580517322318
17 문서 인덱스: 1 유사도: 0.07612205066324826
18 문서 인덱스: 12 유사도: 0.06272935662417738
19 문서 인덱스: 18 유사도: 0.057222817977024304
20 문서 인덱스: 15 유사도: 0.056711171503357694
21 문서 인덱스: 5 유사도: 0.05586888319169245
22 문서 인덱스: 11 유사도: 0.0452495345059608
23 문서 인덱스: 9 유사도: 0.044651454371606586
24 문서 인덱스: 10 유사도: 0.04446240108355923
25 문서 인덱스: 27 유사도: 0.032795082305110315
26 문서 인덱스: 13 유사도: 0.017455403129155535
27 

In [31]:
data['정제본문'][[6,26,29]]

6     교통 호재가 있는 곳에 아파트 분양이 이어지고 있어 주목을 받고 있습니다 주 한라가...
26    선제진입한 대우 롯데이어 시공능력 1 2위 삼성 현대 합류 현대엔지니어링 포스코건설...
29    경제와이드 모닝벨 조간 브리핑 장연재 조간브리핑입니다 발길 끊는 개미들 거래대금 2...
Name: 정제본문, dtype: str

### 실습
1. 주제를 정해서 문서 수집하기(공공데이터 포털 등등)
2. 질문을 했을 경우 유사한 문서를 가져오도록 코드 작성하기
3. 로컬 모델과 openai 임베딩 모델의 차이 비교해보고, 문서 정제 어떻게 할지 고민해보기
4. 결과 이상하면 질문 자체를 바꿔서 해보기! 예를 들어 질문이 더 디테일해야 할 필요

In [51]:
df = pd.read_csv("../data/전국도서관표준데이터.csv", encoding="cp949")
df.columns

Index(['도서관명', '시도명', '시군구명', '도서관유형', '휴관일', '평일운영시작시각', '평일운영종료시각',
       '토요일운영시작시각', '토요일운영종료시각', '공휴일운영시작시각', '공휴일운영종료시각', '열람좌석수', '자료수(도서)',
       '자료수(연속간행물)', '자료수(비도서)', '대출가능권수', '대출가능일수', '소재지도로명주소', '운영기관명',
       '도서관전화번호', '부지면적', '건물면적', '홈페이지주소', '위도', '경도', '데이터기준일자', '제공기관코드',
       '제공기관명'],
      dtype='str')

In [54]:
df.head(3)

,도서관명,시도명,시군구명,도서관유형,휴관일,평일운영시작시각,평일운영종료시각,토요일운영시작시각,토요일운영종료시각,공휴일운영시작시각,...,운영기관명,도서관전화번호,부지면적,건물면적,홈페이지주소,위도,경도,데이터기준일자,제공기관코드,제공기관명
0,전라남도교육청보성도서관,전라남도,보성군,공공도서관,월+법정공휴일+대체공휴일+특별한 사유로 관장이 지정한 날,09:00,18:00,09:00,17:00,09:00,...,전라남도교육청,061-852-3893,4876.0,2642.0,https://bslib.jne.go.kr/index.es?sid=a8,34.771148,127.084561,2026-01-13,7140000,전남광주통합특별시교육청
1,전라남도교육청구례도서관,전라남도,구례군,공공도서관,월+법정공휴일+대체공휴일+특별한 사유로 관장이 지정한 날,09:00,18:00,09:00,17:00,09:00,...,전라남도교육청,061-782-2366,2000.0,1940.0,https://grlib.jne.go.kr/index.es?sid=a7,35.205420,127.465290,2026-01-13,7140000,전남광주통합특별시교육청
2,전라남도교육청곡성교육문화회관,전라남도,곡성군,공공도서관,월+법정공휴일+대체공휴일+특별한 사유로 관장이 지정한 날,09:00,18:00,09:00,17:00,09:00,...,전라남도교육청,061-362-0671,4420.0,3803.0,https://gslib.jne.go.kr/index.es?sid=c1,35.282157,127.289701,2026-01-13,7140000,전남광주통합특별시교육청


In [55]:
columns = ['도서관명', '시도명', '시군구명']

df = df[columns]
df.head(3)

,도서관명,시도명,시군구명
0,전라남도교육청보성도서관,전라남도,보성군
1,전라남도교육청구례도서관,전라남도,구례군
2,전라남도교육청곡성교육문화회관,전라남도,곡성군


In [61]:
df['데이터'] = df.apply(
    lambda row: f"{row['도서관명']}은 {row['시도명']} {row['시군구명']}에 있는 도서관이다.",
    axis=1
)

df.head(3)

,도서관명,시도명,시군구명,데이터
0,전라남도교육청보성도서관,전라남도,보성군,전라남도교육청보성도서관은 전라남도 보성군에 있는 도서관이다.
1,전라남도교육청구례도서관,전라남도,구례군,전라남도교육청구례도서관은 전라남도 구례군에 있는 도서관이다.
2,전라남도교육청곡성교육문화회관,전라남도,곡성군,전라남도교육청곡성교육문화회관은 전라남도 곡성군에 있는 도서관이다.


In [62]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "dragonkue/BGE-m3-ko",
    device="cpu"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [64]:
article_vecs = model.encode(df['데이터'].tolist(), 
                             normalize_embeddings=True,
                             batch_size=4)

In [74]:
article_vecs.shape

(3601, 1024)

In [112]:
from sentence_transformers import util

def search_text(query, k=3):    # 질문과 유사한 문서 최대 k 개 가져오기
    query_vec = model.encode("query: " + query, normalize_embeddings=True)
    similarity = util.cos_sim(query_vec, article_vecs)[0]
    for i in similarity.argsort(descending=True)[:k]:   #유사도 상위 k개만 뽑음
        i = int(i)
        print(f"{similarity[i]}, 그렇다면 {df['시군구명'].iloc[i]}에 있는 {df['도서관명'].iloc[i]}으로 가세요")


In [136]:
query = "나 경남인데 작은 도서관 가고싶어"
search_text(query,5)

0.5563572645187378, 그렇다면 김해시에 있는 한국작은도서관으로 가세요
0.5508335828781128, 그렇다면 김해시에 있는 생각이크는작은도서관으로 가세요
0.5391716957092285, 그렇다면 김해시에 있는 우리작은도서관으로 가세요
0.5029788017272949, 그렇다면 김해시에 있는 소리작은도서관으로 가세요
0.5004709959030151, 그렇다면 김해시에 있는 율하e편한작은도서관으로 가세요
